# Forecast Comparison

## Goal

Merge institutional nowcasts, market-implied GDP distributions, and BEA releases for forecast comparison.

In [ ]:
import pandas as pd

from src.config import EXTERNAL_DATA_DIR, PROCESSED_DATA_DIR
from src.nowcasts import aggregate_institutional_nowcasts, load_institutional_nowcasts

## Load institutional nowcasts

In [ ]:
nowcast_path = EXTERNAL_DATA_DIR / "institutional_nowcasts.csv"
if not nowcast_path.exists():
    raise FileNotFoundError(f"Required input file does not exist: {nowcast_path}")
if nowcast_path.stat().st_size == 0:
    raise ValueError(f"Required input file is empty: {nowcast_path}")

nowcasts = load_institutional_nowcasts(nowcast_path)
if nowcasts["date"].isna().any():
    raise ValueError("Institutional nowcast dates did not parse correctly.")

institutional_panel = aggregate_institutional_nowcasts(nowcasts)
if institutional_panel["institutional_mean"].isna().any():
    raise ValueError("institutional_mean is null after aggregation.")

institutional_panel

## Load market-implied GDP panel

In [ ]:
kalshi_path = PROCESSED_DATA_DIR / "kalshi_q2_2026_distribution_panel.csv"
if not kalshi_path.exists():
    raise FileNotFoundError(f"Required input file does not exist: {kalshi_path}")

market_panel = pd.read_csv(kalshi_path)
market_panel["date"] = pd.to_datetime(market_panel["date"], errors="coerce")
if market_panel["date"].isna().any():
    raise ValueError("Kalshi distribution dates did not parse correctly.")

market_panel = market_panel.rename(
    columns={
        "market_mean_gdp": "kalshi_mean_gdp",
        "market_median_gdp": "kalshi_median_gdp",
        "market_std_gdp": "kalshi_std_gdp",
    }
)
market_panel[["date", "quarter", "kalshi_mean_gdp", "kalshi_median_gdp", "kalshi_std_gdp"]]

## Merge comparison panel

In [ ]:
kalshi_quarters = set(market_panel["quarter"].dropna().astype(str))
institutional_quarters = set(institutional_panel["quarter"].dropna().astype(str))
if not kalshi_quarters.intersection(institutional_quarters):
    raise ValueError(
        "Quarter values do not match between Kalshi and institutional inputs: "
        f"kalshi={sorted(kalshi_quarters)}, institutional={sorted(institutional_quarters)}"
    )

comparison_panel = market_panel.merge(institutional_panel, on=["date", "quarter"], how="inner")
if comparison_panel.empty:
    raise ValueError("No matching date-quarter rows after merging Kalshi and institutional panels.")

comparison_panel["market_minus_institutional"] = (
    comparison_panel["kalshi_mean_gdp"] - comparison_panel["institutional_mean"]
)
if comparison_panel["market_minus_institutional"].isna().any():
    raise ValueError("market_minus_institutional was not computed.")

comparison_panel

## Simple comparison table

In [ ]:
comparison_table = comparison_panel[
    [
        "date",
        "quarter",
        "kalshi_mean_gdp",
        "kalshi_median_gdp",
        "kalshi_std_gdp",
        "institutional_mean",
        "market_minus_institutional",
    ]
].sort_values(["date", "quarter"])

comparison_table

## Save comparison panel

In [ ]:
processed_path = PROCESSED_DATA_DIR / "q2_2026_forecast_comparison_panel.csv"
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
comparison_panel.to_csv(processed_path, index=False)
processed_path